In [14]:
## Import modules
import os,sys
import numpy as np
import geopandas as gpd
import cftime 
import gc
import shapely
import json 
import logging
import glob
import datetime
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
from pathlib import Path
import psutil
import warnings
warnings.filterwarnings('ignore')

# Add the directory containing 'cmct' to the Python path
# Navigate two levels up to reach main CmCt dir
cmct_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))

# Import utilities for this comparison
sys.path.insert(0,cmct_dir)
from cmct.time_utils import check_datarange
from cmct.calving import *
from cmct.calving_modules.interpolation import *
from cmct.calving_modules.json_to_netcdf import *
from cmct.shapefile_utils import *          

# Force initial garbage collection
gc.collect()

# Configure logging to only show errors
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s')

# Memory monitoring function
def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def clear_memory():
    """Aggressive memory clearing"""
    gc.collect()
    print(f"Memory after cleanup: {get_memory_usage():.1f} MB")

In [15]:
# Reload modules to pick up any changes to imports
import importlib
import cmct.calving
import cmct.shapefile_utils
importlib.reload(cmct.shapefile_utils)
importlib.reload(cmct.calving)

# Re-import to ensure functions are available
from cmct.calving import *
from cmct.shapefile_utils import *

In [16]:
# Observation Dataset
# Ice sheet
loc = 'GIS' # 'GIS' or 'AIS'

# Set the observation data dir path
obs_filename = cmct_dir + '/data/calving/observed_icemask_ismip_annual.nc'

# To use aggregation functions for basin 
basin_aggregation = True # IMPORTANT

basin_shapes = cmct_dir + '/data/ne_10m_coastline/ne_10m_coastline.shp'

# ENSEMBLE CONFIGURATION - Multiple model files using glob
# Define the model file template pattern
model_filename_template = cmct_dir + '/test/calving/ensemble/*.nc'

# Find all model files using glob
model_files = glob.glob(model_filename_template)

# Sort the files for consistent ordering
model_files.sort()

# Generate model names from filenames (extract basename without extension)
model_names = [os.path.splitext(os.path.basename(f))[0] for f in model_files]

# Validate that we found model files
if not model_files:
    raise FileNotFoundError(f"No model files found matching pattern: {model_filename_template}")

print(f"Found {len(model_files)} model files:")
for i, (name, file) in enumerate(zip(model_names, model_files), 1):
    print(f"  {i}. {name} -> {os.path.basename(file)}")

# Set time range for comparison
start_year = 2008
end_year = 2012

# List of basins (ex [NW, NE]) to compare if all -> "all"
basin_list = ["NW", "NE", "SE", "SW", "NO", "CW", "unassigned"]

# Output configuration
output_dir = cmct_dir + '/output/ensemble_results/'
os.makedirs(output_dir, exist_ok=True)

# Chunk processing configuration
chunk_size = 2  # Process 2 years at a time to manage memory
memory_threshold = 8000  # MB - trigger cleanup if memory exceeds this

# Optional Configurations 
interpolation_method = 'nearest' # 'nearest', 'linear', 'cubic'
accuracy_calculation_method = 'mean' # 'mean', 'RMS'

colors = {
    'CW': 'blue',
    'NE': 'red', 
    'SE': 'green',
    'SW': 'orange',
    'NO': 'purple',
    'NW': 'brown'
}

print(f"Ensemble processing configured for {len(model_files)} model files")
print(f"Initial memory usage: {get_memory_usage():.1f} MB")

Found 5 model files:
  1. sftgif_B001_hist -> sftgif_B001_hist.nc
  2. sftgif_B002_hist -> sftgif_B002_hist.nc
  3. sftgif_B003_hist -> sftgif_B003_hist.nc
  4. sftgif_B004_hist -> sftgif_B004_hist.nc
  5. sftgif_B005_hist -> sftgif_B005_hist.nc
Ensemble processing configured for 5 model files
Initial memory usage: 284.1 MB


In [17]:
basins = load_basins_exp(cmct_dir, basin_list)
print(basins)

({'NW': <POLYGON ((-48.069 72.595, -48.985 72.468, -49.352 72.423, -49.493 72.406, -...>, 'NE': <POLYGON ((-43.428 78.308, -43.298 78.408, -43.095 78.553, -42.929 78.669, -...>, 'SE': <POLYGON ((-38.039 66.304, -38.034 66.305, -38.015 66.295, -38.028 66.295, -...>, 'SW': <POLYGON ((-50.241 68.223, -50.202 68.235, -50.096 68.247, -49.96 68.26, -49...>, 'NO': <POLYGON ((-29.298 81.111, -29.202 81.111, -29.191 81.11, -29.124 81.102, -2...>, 'CW': <POLYGON ((-39.02 68.884, -39.321 68.781, -39.645 68.665, -39.685 68.649, -3...>}, ['NW', 'NE', 'SE', 'SW', 'NO', 'CW', 'unassigned'])
